[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Medica/open-medical-skills/blob/main/oms-models/notebooks/06_deploy_to_qdrant.ipynb)

# 06 - Deploy Embeddings to Qdrant

**OMS Custom Embedding Model Pipeline - Step 6 of 6**

This notebook takes the best model from the evaluation step, re-embeds all 2,049 tools
(49 OMS skills + 5 plugins + 1,995 ToolUniverse tools), and uploads them to a Qdrant
vector database for production retrieval.

**Process:**
1. Load the winning model from Step 5
2. Load all tool/skill metadata
3. Generate embeddings for all items
4. Connect to Qdrant (local or remote)
5. Create/recreate the collection with proper configuration
6. Upload embeddings with metadata in batches
7. Verify with test queries
8. Compare against old embeddings

---

## 1. Setup & Dependencies

In [ ]:
!pip install -q qdrant-client sentence-transformers torch pyyaml datasets tqdm

In [ ]:
import json
import os
import sys
import uuid
from pathlib import Path
from typing import Any, Optional

import numpy as np
import torch
import yaml
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Best Model

Load the model that performed best in the evaluation step.
If benchmark results are available, automatically select the winner.
Otherwise, default to the fine-tuned EmbeddingGemma.

In [ ]:
# Try to auto-detect best model from benchmark results
best_model_path = None
best_model_name = None

for results_path in [
    Path('benchmarks/results/benchmark_results.json'),
    Path('../benchmarks/results/benchmark_results.json'),
]:
    if results_path.exists():
        with open(results_path) as f:
            bench_results = json.load(f)
        best = max(bench_results, key=lambda r: r.get('Recall@5', 0))
        best_model_name = best['model']
        print(f"Best model from benchmarks: {best_model_name} (Recall@5: {best['Recall@5']:.4f})")
        break

# Map model names to paths
MODEL_PATH_MAP = {
    'oms-toolrag-gemma-v1': ['oms-toolrag-gemma-v1', '../oms-toolrag-gemma-v1'],
    'oms-toolrag-qwen-v1': ['oms-toolrag-qwen-v1', '../oms-toolrag-qwen-v1'],
    'EmbeddingGemma-300M': ['models/embeddinggemma-300m', '../models/embeddinggemma-300m', 'google/embeddinggemma'],
    'ToolRAG-T1': ['models/toolrag-t1', '../models/toolrag-t1', 'yiminzhan/ToolRAG-T1'],
}

# Find the model path
if best_model_name:
    for key, paths in MODEL_PATH_MAP.items():
        if key in best_model_name:
            for p in paths:
                if Path(p).exists() or '/' in p:  # HF hub paths contain '/'
                    best_model_path = p
                    break
            break

# Fallback: use fine-tuned EmbeddingGemma
if best_model_path is None:
    for p in ['oms-toolrag-gemma-v1', '../oms-toolrag-gemma-v1', 'models/embeddinggemma-300m', '../models/embeddinggemma-300m']:
        if Path(p).exists():
            best_model_path = p
            best_model_name = p
            break

assert best_model_path is not None, "No model found. Run notebooks 03/04 first."
print(f"Loading model: {best_model_path}")

In [ ]:
model = SentenceTransformer(best_model_path, trust_remote_code=True)

EMBEDDING_DIM = model.get_sentence_embedding_dimension()
print(f"Model loaded: {best_model_name}")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"Max sequence length: {model.max_seq_length}")

## 3. Load All Tools

Load all 2,049 tools with their full metadata from the OMS repository.

In [ ]:
# Find the repo root
REPO_ROOT = Path('.').resolve()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'content' / 'skills').exists():
        REPO_ROOT = parent
        break

# Add scripts to path
scripts_dir = REPO_ROOT / 'oms-models' / 'data' / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

try:
    from generate_synthetic_queries import (
        load_oms_skills,
        load_oms_plugins,
        load_tu_tools,
        build_item_description,
        get_item_name,
        get_item_category,
    )

    skills = load_oms_skills()
    plugins = load_oms_plugins()
    tu_tools = load_tu_tools()
    all_items = skills + plugins + tu_tools

    print(f"Loaded from repo scripts:")
    print(f"  OMS Skills:  {len(skills)}")
    print(f"  OMS Plugins: {len(plugins)}")
    print(f"  TU Tools:    {len(tu_tools)}")
    print(f"  Total:       {len(all_items)}")

except ImportError:
    print("Could not import data loaders. Loading from processed dataset instead.")

    # Fallback: build items from the training dataset
    all_items = []
    from datasets import load_from_disk

    for data_path in [
        Path('data/processed/oms_training_pairs'),
        Path('../data/processed/oms_training_pairs'),
    ]:
        if data_path.exists():
            ds = load_from_disk(str(data_path))
            seen_tools = {}
            for split_name in ds:
                for r in ds[split_name]:
                    tool_name = r['tool_name']
                    if tool_name not in seen_tools:
                        seen_tools[tool_name] = {
                            'name': tool_name,
                            'display_name': tool_name,
                            'description': r.get('tool_description', ''),
                            '_source': r.get('source', 'unknown'),
                            'category': r.get('category', 'uncategorized'),
                        }
            all_items = list(seen_tools.values())
            print(f"Loaded {len(all_items)} unique tools from processed dataset.")
            break

    # Define helper functions
    def build_item_description(item):
        return f"Name: {item.get('display_name', item.get('name', 'Unknown'))}\nDescription: {item.get('description', 'N/A')}\nCategory: {item.get('category', 'N/A')}"

    def get_item_name(item):
        return item.get('display_name', item.get('name', 'Unknown'))

    def get_item_category(item):
        return item.get('category', 'uncategorized')

assert len(all_items) > 0, "No items loaded."

## 4. Generate Embeddings

Encode all tool descriptions in batches. This produces the vectors that will be stored in Qdrant.

In [ ]:
# Build descriptions for all items
descriptions = [build_item_description(item) for item in all_items]
names = [get_item_name(item) for item in all_items]
categories = [get_item_category(item) for item in all_items]
sources = [item.get('_source', 'unknown') for item in all_items]

print(f"Encoding {len(descriptions)} tool descriptions...")
print(f"Embedding dimension: {EMBEDDING_DIM}")

# Batch encode
ENCODE_BATCH_SIZE = 64
embeddings = model.encode(
    descriptions,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=ENCODE_BATCH_SIZE,
    normalize_embeddings=True,  # L2 normalize for cosine similarity
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Embeddings dtype: {embeddings.dtype}")
print(f"L2 norm (sample): {np.linalg.norm(embeddings[0]):.4f} (should be ~1.0)")

## 5. Connect to Qdrant

Configure the Qdrant client. Supports:
- **Local in-memory** (for testing)
- **Local file-based** (persistent, no server needed)
- **Remote server** (production deployment)
- **Qdrant Cloud** (managed service)

In [ ]:
# Qdrant connection configuration
# Uncomment the appropriate option:

# Option 1: Local in-memory (for testing - data lost when notebook stops)
qdrant = QdrantClient(location=":memory:")
QDRANT_MODE = 'memory'

# Option 2: Local file-based (persistent, no server needed)
# qdrant = QdrantClient(path="./qdrant_data")
# QDRANT_MODE = 'local'

# Option 3: Remote Qdrant server
# QDRANT_URL = os.getenv('QDRANT_URL', 'http://localhost:6333')
# QDRANT_API_KEY = os.getenv('QDRANT_API_KEY', None)
# qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
# QDRANT_MODE = 'remote'

# Option 4: Qdrant Cloud
# qdrant = QdrantClient(
#     url='https://your-cluster.cloud.qdrant.io:6333',
#     api_key=os.getenv('QDRANT_CLOUD_API_KEY'),
# )
# QDRANT_MODE = 'cloud'

print(f"Qdrant mode: {QDRANT_MODE}")
print(f"Connected to Qdrant.")

## 6. Create Collection

Create (or recreate) the `oms_tools_custom` collection with the appropriate vector configuration.
Uses cosine distance since embeddings are normalized.

In [ ]:
COLLECTION_NAME = 'oms_tools_custom'

# Delete existing collection if it exists
try:
    qdrant.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")
except Exception:
    pass

# Create collection with proper configuration
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIM,
        distance=models.Distance.COSINE,
        on_disk=True if QDRANT_MODE != 'memory' else False,
    ),
    # Optimize for search performance
    optimizers_config=models.OptimizersConfigDiff(
        indexing_threshold=1000,  # Build HNSW index after 1000 points
    ),
    # HNSW index params
    hnsw_config=models.HnswConfigDiff(
        m=16,
        ef_construct=100,
    ),
)

print(f"Created collection: {COLLECTION_NAME}")
print(f"  Vector dimension: {EMBEDDING_DIM}")
print(f"  Distance metric: Cosine")

# Verify
info = qdrant.get_collection(COLLECTION_NAME)
print(f"  Status: {info.status}")
print(f"  Points: {info.points_count}")

## 7. Upload Points

Upload all embeddings with their metadata (tool name, category, source, description, etc.) in batches.

In [ ]:
UPLOAD_BATCH_SIZE = 100

# Build points with metadata
points = []
for i, item in enumerate(all_items):
    # Build payload (metadata stored alongside the vector)
    payload = {
        'name': get_item_name(item),
        'source': item.get('_source', 'unknown'),
        'category': get_item_category(item),
        'description': item.get('description', '')[:500],  # Truncate for storage
    }

    # Add OMS-specific fields
    if item.get('_source') in ('oms_skill', 'oms_plugin'):
        payload.update({
            'display_name': item.get('display_name', ''),
            'author': item.get('author', ''),
            'evidence_level': item.get('evidence_level', ''),
            'safety': item.get('safety', item.get('safety_classification', '')),
            'tags': item.get('tags', []),
            'repository': item.get('repository', ''),
            'verified': item.get('verified', False),
            'reviewer': item.get('reviewer', ''),
        })

    # Add TU-specific fields
    if item.get('_source') == 'tu_tool':
        payload.update({
            'tool_type': item.get('type', ''),
            'has_parameters': bool(item.get('parameter', {})),
        })

    points.append(models.PointStruct(
        id=i,  # Use sequential integer IDs
        vector=embeddings[i].tolist(),
        payload=payload,
    ))

# Upload in batches
total_uploaded = 0
for batch_start in tqdm(range(0, len(points), UPLOAD_BATCH_SIZE), desc="Uploading batches"):
    batch = points[batch_start:batch_start + UPLOAD_BATCH_SIZE]
    qdrant.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
    )
    total_uploaded += len(batch)

print(f"\nUpload complete!")
print(f"Total points uploaded: {total_uploaded}")

# Verify count
info = qdrant.get_collection(COLLECTION_NAME)
print(f"Collection points: {info.points_count}")
assert info.points_count == len(all_items), f"Mismatch: {info.points_count} != {len(all_items)}"

## 8. Verify with Test Queries

Run 5 test queries against the deployed collection and display the top-5 results for each.

In [ ]:
# Define test queries spanning different medical categories
test_queries_verify = [
    "tool for checking drug interactions in elderly patients",
    "AI skill for analyzing chest X-ray findings",
    "calculate pediatric drug dosing based on weight",
    "summarize clinical trial results for oncology research",
    "prior authorization documentation generator",
]

print(f"{'='*80}")
print(f"VERIFICATION: Test Queries Against Qdrant")
print(f"{'='*80}")

for query in test_queries_verify:
    # Encode the query
    q_embedding = model.encode(query, normalize_embeddings=True).tolist()

    # Search Qdrant
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embedding,
        limit=5,
        with_payload=True,
    )

    print(f"\nQuery: \"{query}\"")
    print(f"  Top-5 results:")
    for rank, point in enumerate(results.points, 1):
        name = point.payload.get('name', 'Unknown')
        source = point.payload.get('source', '?')
        category = point.payload.get('category', '?')
        score = point.score
        print(f"    {rank}. [{source}] {name} ({category}) -- score: {score:.4f}")

## 9. Compare with Old Embeddings (Optional)

If you have an existing collection with embeddings from a different model (e.g., nomic-embed-text
or mxbai-embed-large), compare the search results side by side.

In [ ]:
# This cell compares against an existing collection if available.
# Set OLD_COLLECTION_NAME to your previous collection name.

OLD_COLLECTION_NAME = 'oms_tools'  # Change to your existing collection name

try:
    old_info = qdrant.get_collection(OLD_COLLECTION_NAME)
    print(f"Old collection '{OLD_COLLECTION_NAME}' found: {old_info.points_count} points")

    compare_queries = [
        "check drug interactions for warfarin",
        "mental health screening tool",
        "FHIR resource validation",
    ]

    print(f"\n{'='*80}")
    print(f"SIDE-BY-SIDE COMPARISON")
    print(f"{'='*80}")

    for query in compare_queries:
        q_embedding = model.encode(query, normalize_embeddings=True).tolist()

        new_results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=q_embedding,
            limit=5,
            with_payload=True,
        )

        # For old collection, we would need the old model's embeddings
        # This is a simplified comparison showing just the new results
        print(f"\nQuery: \"{query}\"")
        print(f"  New model ({COLLECTION_NAME}):")
        for rank, point in enumerate(new_results.points, 1):
            name = point.payload.get('name', 'Unknown')
            score = point.score
            print(f"    {rank}. {name} (score: {score:.4f})")

except Exception as e:
    print(f"Old collection '{OLD_COLLECTION_NAME}' not available: {e}")
    print("Skipping comparison. To compare, ensure both collections exist in the same Qdrant instance.")

## 10. Integration Notes

How to connect the deployed Qdrant collection to the OMS website and CLI.

In [ ]:
# Print integration instructions
print(f"""
{'='*80}
INTEGRATION GUIDE
{'='*80}

Collection: {COLLECTION_NAME}
Embedding model: {best_model_name}
Embedding dimension: {EMBEDDING_DIM}
Total points: {len(all_items)}

--- OMS Website (Search API) ---

In workers/search-api/src/index.ts, update the search handler:

  const COLLECTION = '{COLLECTION_NAME}';
  const EMBEDDING_DIM = {EMBEDDING_DIM};

The search API worker needs to:
  1. Receive a search query from the frontend
  2. Encode the query using the same model (or a compatible API)
  3. Query Qdrant with the embedding vector
  4. Return the top-K results with metadata

--- OMS CLI ---

In cli/lib/skills.js, update the search function:

  const QDRANT_URL = 'http://your-qdrant-host:6333';
  const COLLECTION = '{COLLECTION_NAME}';

--- Python Direct Access ---

```python
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('{best_model_path}')
qdrant = QdrantClient(url='http://your-qdrant-host:6333')

query = "find a drug interaction checker"
embedding = model.encode(query, normalize_embeddings=True).tolist()

results = qdrant.query_points(
    collection_name='{COLLECTION_NAME}',
    query=embedding,
    limit=10,
    with_payload=True,
)

for point in results.points:
    print(f"{{point.payload['name']}} (score: {{point.score:.4f}})")```

--- Deployment Options ---

1. Local: qdrant/qdrant Docker container
   docker run -p 6333:6333 -v ./qdrant_data:/qdrant/storage qdrant/qdrant

2. Kubernetes: Deploy qdrant/qdrant to your K3s cluster (namespace: oms)

3. Qdrant Cloud: Managed service at cloud.qdrant.io
   Free tier: 1GB storage, 1M vectors
""")

In [ ]:
# Save embeddings to disk for backup/portability
EXPORT_DIR = Path('deployment/embeddings')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save embeddings as numpy array
np.save(EXPORT_DIR / 'tool_embeddings.npy', embeddings)

# Save metadata as JSONL
with open(EXPORT_DIR / 'tool_metadata.jsonl', 'w') as f:
    for i, item in enumerate(all_items):
        meta = {
            'id': i,
            'name': get_item_name(item),
            'source': item.get('_source', 'unknown'),
            'category': get_item_category(item),
            'description': item.get('description', '')[:500],
        }
        f.write(json.dumps(meta, ensure_ascii=False) + '\n')

# Save collection config
config = {
    'collection_name': COLLECTION_NAME,
    'model_name': best_model_name,
    'model_path': str(best_model_path),
    'embedding_dim': EMBEDDING_DIM,
    'num_points': len(all_items),
    'distance_metric': 'cosine',
    'normalized': True,
}
with open(EXPORT_DIR / 'collection_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Embeddings exported to: {EXPORT_DIR}")
print(f"  tool_embeddings.npy: {embeddings.shape} ({embeddings.nbytes / 1e6:.1f} MB)")
print(f"  tool_metadata.jsonl: {len(all_items)} entries")
print(f"  collection_config.json: deployment configuration")

---

## Pipeline Complete

The OMS custom embedding model pipeline is now complete. Summary of what was accomplished:

1. **Data Generation** (notebook 01): Generated synthetic (query, tool) training pairs
2. **Data Validation** (notebook 02): Multi-model consensus scoring and quality filtering
3. **EmbeddingGemma Training** (notebook 03): Fine-tuned 300M model with CachedMNRL
4. **GTE-Qwen2 Training** (notebook 04): Fine-tuned 1.5B model with LoRA/full
5. **Evaluation** (notebook 05): Benchmarked all models on OMS test set
6. **Deployment** (notebook 06): Deployed best model's embeddings to Qdrant

**Files produced by this notebook:**
- `deployment/embeddings/tool_embeddings.npy` -- All embedding vectors
- `deployment/embeddings/tool_metadata.jsonl` -- Tool metadata
- `deployment/embeddings/collection_config.json` -- Qdrant collection config

**Next steps:**
- Deploy Qdrant to K8s cluster (namespace: oms)
- Update search-api worker to use custom embeddings
- Update CLI search to query Qdrant
- Set up automated re-embedding pipeline for new skills/plugins